In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import importlib.util,json,subprocess,sys,time
import numpy as np
import pandas as pd

INPUT,WORK=Path('/kaggle/input'),Path('/kaggle/working')
if importlib.util.find_spec('numpyro') is None:
    print("Installing NumPyro without changing Kaggle's JAX/CUDA packages...",flush=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-deps','numpyro==0.22.0'],check=True)
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC,NUTS
from numpyro.infer.initialization import init_to_value
from numpyro.diagnostics import summary as numpyro_summary
import matplotlib.pyplot as plt

print('JAX backend:',jax.default_backend(),'| devices:',jax.devices(),'| NumPyro:',numpyro.__version__)
if jax.default_backend()!='gpu': raise RuntimeError('GPU preflight failed: select GPU T4 x2 and restart the session.')
def newest(pattern,label):
    paths=list(INPUT.rglob(pattern))
    if not paths: raise FileNotFoundError(f'Attach saved output containing {label}: {pattern}')
    return max(paths,key=lambda p:p.stat().st_mtime)

binary_sources=[]
for mp in INPUT.rglob('simulation_metadata.json'):
    if mp.parent.name!='binary': continue
    try: meta=json.loads(mp.read_text(encoding='utf-8'))
    except Exception: continue
    if meta.get('family')=='binary' and int(meta.get('parameter_draws_unique',0))>=100_000:
        binary_sources.append((meta.get('created_utc',''),mp.parent))
if not binary_sources: raise FileNotFoundError('Attach Day 7 Cosmic swan Bayesian binary simulations.')
_,BINARY_DIR=sorted(binary_sources,key=lambda x:x[0])[-1]
observations=np.load(BINARY_DIR/'observations.npy',mmap_mode='r')
parameters=pd.read_csv(BINARY_DIR/'parameters.csv')
time_days=np.asarray(np.load(BINARY_DIR/'time_days.npy'),dtype=float)
if observations.ndim!=2 or observations.shape!=(len(parameters),len(time_days)):
    raise ValueError('Day 7 observations, parameters, and time axis do not match.')

MAP_PATH=newest('day9_12_conditional_mle_parameters.csv','Day 9.12 conditional MLE parameters')
MAP_DIAG_PATH=newest('day9_12_conditional_mle_diagnostics.csv','Day 9.12 conditional MLE diagnostics')
PROFILE_PATH=newest('day9_15_nuisance_profile_summary.csv','Day 9.15 nuisance profile summary')
map_df=pd.read_csv(MAP_PATH)
map_diag=pd.read_csv(MAP_DIAG_PATH)
profiles=pd.read_csv(PROFILE_PATH)
ROWS=[87483,4923]
SEEDS={}
for row in ROWS:
    sub=map_df[map_df.simulation_row.astype(int).eq(row)]
    if sub.parameter.duplicated().any(): raise ValueError(f'Duplicate MLE estimates for row {row}.')
    vals=sub.set_index('parameter').estimate.astype(float).to_dict()
    needed={'baseline_flux','phase','depth_primary','depth_ratio','duration_fraction',
        'ellipsoidal_amplitude','reflection_amplitude','noise_sigma'}
    missing=needed-set(vals)
    if missing or any(not np.isfinite(vals[k]) for k in needed if k in vals):
        raise ValueError(f'Invalid MLE values for row {row}: {sorted(missing)}')
    match=map_diag[map_diag.simulation_row.astype(int).eq(row)]
    if len(match)!=1 or not bool(match.iloc[0].optimizer_success): raise ValueError(f'Day 9.12 MLE not successful for row {row}.')
    vals['period_fixed_days']=float(match.iloc[0].period_fixed_days)
    SEEDS[row]=vals
row87483=profiles[profiles.simulation_row.astype(int).eq(87483)&profiles.profile_parameter.eq('duration_fraction')]
if len(row87483)!=1: raise ValueError('Day 9.15 duration profile for row 87483 is missing or duplicated.')
DURATION_FIXED=float(row87483.iloc[0].profile_minimum)
if not 0.005<=DURATION_FIXED<=0.12: raise ValueError(f'Invalid fixed duration: {DURATION_FIXED}')
if max(ROWS)>=len(parameters): raise ValueError('A selected row is outside the Day 7 simulation table.')
OUT=WORK/('day9-16-targeted-conditional-nuts-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True,exist_ok=False)
CHAINS,TARGET_ACCEPT,MAX_TREE_DEPTH=2,0.90,10
CHAIN_METHOD='parallel' if len(jax.devices())>=2 else 'sequential'
print('[PASS] Day 7:',BINARY_DIR)
print('[PASS] Day 9.12 MLE estimates:',MAP_PATH.parent)
print('[PASS] Day 9.15 profiles:',PROFILE_PATH.parent)
print('Rows:',ROWS,'| row 87483 fixed duration:',DURATION_FIXED,'| period values:',{r:SEEDS[r]['period_fixed_days'] for r in ROWS})
print('Output:',OUT)


In [ ]:
def smooth_clip_jax(x,sharpness=500.0):
    return (jax.nn.softplus(sharpness*x)-jax.nn.softplus(sharpness*(x-1.0)))/sharpness
def predict_jax(t,period,phase,depth,ratio,duration,ellip,reflection,baseline):
    orb=jnp.mod(t/period-phase+0.5,1.0)-0.5
    sec=jnp.mod(orb,1.0)-0.5
    half=jnp.maximum(duration/2.0,1e-6)
    ingress=jnp.maximum(half*0.18,1e-6)
    primary=smooth_clip_jax((half+ingress-jnp.abs(orb))/(2.0*ingress))
    secondary=smooth_clip_jax((half+ingress-jnp.abs(sec))/(2.0*ingress))
    return baseline-depth*primary-depth*ratio*secondary+ellip*jnp.cos(4*jnp.pi*orb)+reflection*jnp.cos(2*jnp.pi*orb)

case_rows=[]
parameter_rows=[]
interval_rows=[]
plot_rows=[]
for row in ROWS:
    started=time.time()
    seeds=SEEDS[row]
    period=float(seeds['period_fixed_days'])
    duration_is_fixed=(row==87483)
    duration_fixed=float(DURATION_FIXED) if duration_is_fixed else None
    warmup,samples=(500,800) if duration_is_fixed else (500,1200)
    t=np.asarray(time_days,dtype=float)
    y=np.asarray(observations[row],dtype=float)
    keep=np.isfinite(t)&np.isfinite(y)
    t,y=t[keep],y[keep]
    order=np.argsort(t)
    t,y=t[order],y[order]
    t=t-t.min()
    phase0=float(seeds['phase']%1.0)
    time_j,obs_j=jnp.asarray(t),jnp.asarray(y)

    def model():
        baseline=numpyro.sample('baseline_flux',dist.Normal(1.0,0.02))
        phase_offset=numpyro.sample('phase_offset',dist.Normal(0.0,0.025))
        depth=numpyro.sample('depth_primary',dist.Uniform(0.002,0.45))
        ratio=numpyro.sample('depth_ratio',dist.Uniform(0.0,1.0))
        if duration_is_fixed:
            duration=jnp.asarray(duration_fixed)
        else:
            duration=numpyro.sample('duration_fraction',dist.Uniform(0.005,0.12))
        ellip=numpyro.sample('ellipsoidal_amplitude',dist.Uniform(0.0,0.02))
        reflection=numpyro.sample('reflection_amplitude',dist.Uniform(-0.01,0.01))
        log_noise=numpyro.sample('log_noise',dist.Uniform(np.log(1e-4),np.log(0.03)))
        phase=jnp.mod(phase0+phase_offset,1.0)
        pred=predict_jax(time_j,period,phase,depth,ratio,duration,ellip,reflection,baseline)
        numpyro.sample('flux',dist.Normal(pred,jnp.exp(log_noise)),obs=obs_j)

    init={'baseline_flux':float(np.clip(seeds['baseline_flux'],0.9501,1.0499)),
        'phase_offset':0.0,
        'depth_primary':float(np.clip(seeds['depth_primary'],0.002001,0.449999)),
        'depth_ratio':float(np.clip(seeds['depth_ratio'],1e-5,0.99999)),
        'ellipsoidal_amplitude':float(np.clip(seeds['ellipsoidal_amplitude'],1e-5,0.01999)),
        'reflection_amplitude':float(np.clip(seeds['reflection_amplitude'],-0.00999,0.00999)),
        'log_noise':float(np.log(np.clip(seeds['noise_sigma'],1.0001e-4,0.02999)))}
    if not duration_is_fixed:
        init['duration_fraction']=float(np.clip(seeds['duration_fraction'],0.005001,0.119999))
    latent=tuple(init.keys())
    kernel=NUTS(model,init_strategy=init_to_value(values=init),target_accept_prob=TARGET_ACCEPT,
        max_tree_depth=MAX_TREE_DEPTH,dense_mass=[latent])
    mcmc=MCMC(kernel,num_warmup=warmup,num_samples=samples,num_chains=CHAINS,
        chain_method=CHAIN_METHOD,progress_bar=True)
    try:
        mcmc.run(jax.random.PRNGKey(160000+row),extra_fields=('diverging','accept_prob','num_steps'))
        grouped=mcmc.get_samples(group_by_chain=True)
        diag=numpyro_summary(grouped,group_by_chain=True)
        per_param=[]
        for name,item in diag.items():
            r=np.asarray(item.get('r_hat',np.nan),dtype=float)
            e=np.asarray(item.get('n_eff',np.nan),dtype=float)
            rh=float(np.nanmax(r)) if np.isfinite(r).any() else float('nan')
            es=float(np.nanmin(e)) if np.isfinite(e).any() else float('nan')
            per_param.append((name,rh,es))
        extras=mcmc.get_extra_fields(group_by_chain=True)
        div=int(np.asarray(extras['diverging']).sum())
        steps=np.asarray(extras['num_steps'],dtype=float)
        accept=np.asarray(extras['accept_prob'],dtype=float)
        rhats=[x[1] for x in per_param if np.isfinite(x[1])]
        esses=[x[2] for x in per_param if np.isfinite(x[2])]
        maxr=max(rhats) if rhats else float('nan')
        mine=min(esses) if esses else float('nan')
        status='screen_pass' if len(rhats)==len(per_param) and len(esses)==len(per_param) and maxr<=1.05 and mine>=100 and div==0 else 'needs_review'
        samples_by_chain={k:np.asarray(v) for k,v in grouped.items()}
        samples_by_chain['phase']=(phase0+samples_by_chain['phase_offset'])%1.0
        samples_by_chain['noise_sigma']=np.exp(samples_by_chain['log_noise'])
        if duration_is_fixed:
            samples_by_chain['duration_fraction']=np.full((CHAINS,samples),duration_fixed,dtype=float)
        np.savez_compressed(OUT/f'row_{row}_posterior_by_chain.npz',**samples_by_chain)
        truth=parameters.iloc[row]
        truth_map={name:float(truth[name]) for name in ['phase','depth_primary','depth_ratio','duration_fraction',
            'ellipsoidal_amplitude','reflection_amplitude','noise_sigma']}
        truth_map['baseline_flux']=1.0
        phase_truth=(truth_map['phase']-phase0+0.5)%1.0-0.5
        for name,rhat,ess in per_param:
            parameter_rows.append({'simulation_row':row,'parameter':name,'r_hat':rhat,'effective_sample_size':ess})
        derived={'baseline_flux':samples_by_chain['baseline_flux'],'phase_offset':samples_by_chain['phase_offset'],
            'depth_primary':samples_by_chain['depth_primary'],'depth_ratio':samples_by_chain['depth_ratio'],
            'duration_fraction':samples_by_chain.get('duration_fraction'),
            'ellipsoidal_amplitude':samples_by_chain['ellipsoidal_amplitude'],
            'reflection_amplitude':samples_by_chain['reflection_amplitude'],'noise_sigma':samples_by_chain['noise_sigma']}
        for name,arr in derived.items():
            if arr is None: continue
            q05,med,q95=np.quantile(arr,[0.05,0.5,0.95])
            target=phase_truth if name=='phase_offset' else truth_map[name]
            interval_rows.append({'simulation_row':row,'parameter':name,'period_fixed_days':period,
                'duration_fixed':duration_is_fixed,'duration_fixed_days_fraction':duration_fixed,
                'q05':float(q05),'median':float(med),'q95':float(q95),
                'known_truth_evaluation_only':float(target),'truth_inside_90pct_interval':bool(q05<=target<=q95),
                'scope':'conditional_on_profile_period_and_duration' if duration_is_fixed else 'conditional_on_profile_period'})
        med={k:np.median(v,axis=(0,1)) for k,v in samples_by_chain.items()}
        pred=np.asarray(predict_jax(time_j,period,float((phase0+med['phase_offset'])%1.0),
            float(med['depth_primary']),float(med['depth_ratio']),duration_fixed if duration_is_fixed else float(med['duration_fraction']),
            float(med['ellipsoidal_amplitude']),float(med['reflection_amplitude']),float(med['baseline_flux'])),dtype=float)
        plot_rows.append((row,t,y,pred,period,duration_fixed))
        case_rows.append({'simulation_row':row,'status':status,'period_fixed_days':period,
            'duration_fixed':duration_is_fixed,'duration_fixed_value':duration_fixed,
            'chains':CHAINS,'warmup_per_chain':warmup,'samples_per_chain':samples,
            'max_r_hat':maxr,'min_effective_sample_size':mine,'divergences':div,
            'mean_accept_probability':float(np.nanmean(accept)),'median_leapfrog_steps':float(np.nanmedian(steps)),
            'fraction_at_max_tree_depth':float(np.mean(steps >= (2**MAX_TREE_DEPTH-1))),
            'seconds':round(time.time()-started,1)})
    except Exception as exc:
        case_rows.append({'simulation_row':row,'status':'failed','period_fixed_days':period,
            'duration_fixed':duration_is_fixed,'duration_fixed_value':duration_fixed,
            'error':f'{type(exc).__name__}: {exc}','seconds':round(time.time()-started,1)})
    pd.DataFrame(case_rows).to_csv(OUT/'day9_16_case_diagnostics.csv',index=False)
    pd.DataFrame(parameter_rows).to_csv(OUT/'day9_16_per_parameter_diagnostics.csv',index=False)
    pd.DataFrame(interval_rows).to_csv(OUT/'day9_16_conditional_intervals.csv',index=False)
    print(f'Targeted conditional NUTS row {row}: {case_rows[-1]}',flush=True)


In [ ]:
if plot_rows:
    fig,axes=plt.subplots(1,len(plot_rows),figsize=(14,5),squeeze=False,constrained_layout=True)
    for ax,(row,t,y,pred,period,duration_fixed) in zip(axes.flat,plot_rows):
        ax.scatter(t,y,s=5,alpha=0.3,label='synthetic observations')
        ax.plot(t,pred,color='crimson',lw=0.9,label='conditional posterior median fit')
        suffix=f' | fixed duration {duration_fixed:.6f}' if duration_fixed is not None else ''
        ax.set_title(f'Row {row} | fixed period {period:.6f} d{suffix}')
        ax.set_xlabel('Time (days)'); ax.set_ylabel('Flux'); ax.legend(fontsize=8)
    fig.savefig(OUT/'day9_16_conditional_fits.png',dpi=140)
    plt.show()

case_df=pd.DataFrame(case_rows)
param_df=pd.DataFrame(parameter_rows)
interval_df=pd.DataFrame(interval_rows)
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'day9_12_input':str(MAP_PATH.parent),
    'day9_15_input':str(PROFILE_PATH.parent),'rows':ROWS,
    'cases':case_df.to_dict(orient='records'),
    'limits':['Row 87483 conditions on both profile period and profile duration; it has no duration posterior interval.',
        'Row 4923 conditions on period only; noise remains sampled.',
        'Intervals and truth checks are conditional synthetic diagnostics, not calibration.',
        'Known truth is evaluation-only and is not used in fitting.',
        'A screen-pass does not establish real-star validity.'],
    'outputs':['day9_16_case_diagnostics.csv','day9_16_per_parameter_diagnostics.csv',
        'day9_16_conditional_intervals.csv','day9_16_conditional_fits.png','row_*_posterior_by_chain.npz']}
(OUT/'day9_16_targeted_nuts_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(case_df.to_string(index=False))
print('Per-parameter diagnostics:')
print(param_df.to_string(index=False))
print(json.dumps(summary,indent=2))
print('Outputs:',OUT)